# Module 1 — Segmentation Training (DRG-Net, SMP U-Net, FGADR)

Runs DRG-Net's own `dr_segmentation/train_fgadr.py` unmodified, with our reduced-epoch
config (`module1/configs/config_fgadr_seg_poc.py`) swapped in for `config_fgadr.py`.

**Epochs are deliberately reduced from the paper's published 1500 to 25**, and only 2 of 4
lesion classes (EX, MA) are trained tonight -- HE/SE are the same script, just rerun with
`--lesion HE` / `--lesion SE`, as an immediate next step. This is the honest, direct answer
to "why not 1500 epochs" (b.3) -- say so plainly in the presentation, don't present this as
matching the paper's published segmentation performance.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os
# This repo is PRIVATE -- Colab needs a GitHub Personal Access Token to clone it.
# Create one (once, reusable across all 4 notebooks/sessions) at
# https://github.com/settings/tokens -> "Generate new token (classic)" -> scope: repo.
# Input is hidden; not saved anywhere.
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'  # <-- change if you used a different folder
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR} -- upload the 4 dataset zips there first"


In [ ]:
# Unzip datasets locally on the Colab VM disk (much faster I/O than reading zips off Drive
# directly). -n skips files that already exist, so this is safe/cheap to re-run.
os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FGADR-Seg-set_Release.zip"
!unzip -q -n "$DRIVE_DATA_DIR/archive.zip" -d IDRiD_dataset_root
!unzip -q -n "$DRIVE_DATA_DIR/FIRE_dataset.zip"
!unzip -q -n "$DRIVE_DATA_DIR/LongDRScreening_20150209.zip"

# archive.zip may unzip with an extra nesting level; normalize so IDRiD_dataset ends up
# directly under /content/data
import glob, shutil
candidates = glob.glob('/content/data/IDRiD_dataset_root/**/IDRiD_dataset', recursive=True)
if candidates and not os.path.isdir('/content/data/IDRiD_dataset'):
    shutil.move(candidates[0], '/content/data/IDRiD_dataset')
print('IDRiD_dataset present:', os.path.isdir('/content/data/IDRiD_dataset'))


In [ ]:
# Clone this repo (private -- uses the token above) and the pinned DRG-Net reference
# implementation. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir('/content/M2-DRProgression'):
    !git clone --branch module1-fgadr-poc https://{GITHUB_TOKEN}@github.com/bearawr/M2-DRProgression.git M2-DRProgression
if not os.path.isdir('/content/dr-joint-learning'):
    !git clone https://github.com/DFKI-Interactive-Machine-Learning/dr-joint-learning.git dr-joint-learning
    %cd dr-joint-learning
    !git checkout 0da1bbe885f438390a0e94ec486283d9b21d5547
    %cd /content

# Compatibility patch: DRG-Net's own code (dr_classification/data/transforms.py) still uses
# torchvision.transforms.RandomAffine's old 'fillcolor' kwarg, renamed to 'fill' and fully
# removed in the torchvision version Colab now ships (we deliberately don't install the repo's
# old pinned torchvision -- see notebook 02's install cell -- so this old kwarg name breaks).
# Idempotent: sed -i only changes the file if 'fillcolor' is still present.
!sed -i 's/fillcolor=aug_args.value_fill/fill=aug_args.value_fill/' /content/dr-joint-learning/dr_classification/data/transforms.py

# Same class of issue, in dr_segmentation's transform code (only exercised by notebook 03,
# harmless to run here too): functional.py imports PILLOW_VERSION, removed from Pillow years
# ago; transforms_group.py bare-imports ipdb, a debugger package not installed by default.
# Both would crash the import chain immediately (from transform.transforms_group import *).
!pip install -q ipdb
_func_py = '/content/dr-joint-learning/dr_segmentation/transform/functional.py'
_txt = open(_func_py).read()
_txt = _txt.replace(
    'from PIL import Image, ImageOps, ImageEnhance, PILLOW_VERSION',
    "from PIL import Image, ImageOps, ImageEnhance\nPILLOW_VERSION = '9.0.0'"
)
open(_func_py, 'w').write(_txt)

# Checkpoint cadence: train_fgadr.py only saves at `(epoch + 1) % 20 == 0` -- one save point
# every 20 epochs, tuned for the paper's 1500-epoch runs. At our POC scale (EPOCHES=25) that
# fires just once, at epoch 20, and not at all if the run is cut short before then (Colab
# reclamation, manual interrupt). Drop it to every 5 epochs so a checkpoint lands regularly
# and at the final epoch. Idempotent: only rewrites while the original '% 20' line is present.
!sed -i 's/(epoch + 1) % 20 == 0/(epoch + 1) % 5 == 0/' /content/dr-joint-learning/dr_segmentation/train_fgadr.py

# Belt-and-suspenders: the checkpoint filename is built as 'model_' + preprocess + '.pth.tar'.
# If argparse hands `preprocess` back as an int, that line TypeErrors at the first save and
# kills the whole run. str() makes it safe either way; no-op if it's already a string.
!sed -i "s/'model_' + preprocess + '.pth.tar'/'model_' + str(preprocess) + '.pth.tar'/" /content/dr-joint-learning/dr_segmentation/train_fgadr.py

!grep -n "epoch + 1) %\|'model_' + " /content/dr-joint-learning/dr_segmentation/train_fgadr.py


In [ ]:
# Copy DRG-Net's own filtered-label CSV (vendored in our repo) into the local FGADR folder --
# dr_segmentation/utils.py::get_images_fgadr_from_pd reads this file from inside IMAGE_DIR.
# Made robust to zip-nesting variation (some zip tools add/drop the top-level
# FGADR-Seg-set_Release wrapper folder) by locating Seg-set/ wherever it actually landed.
import glob as _glob, os, shutil

_candidates = _glob.glob('/content/data/**/Seg-set', recursive=True)
assert _candidates, (
    'No Seg-set folder found under /content/data. Run: !ls /content/data  and  '
    '!ls "$DRIVE_DATA_DIR"  to check the zip is named exactly FGADR-Seg-set_Release.zip '
    'and actually unzipped in the previous cell.'
)
fgadr_segset_dir = _candidates[0]
print('Found FGADR Seg-set at:', fgadr_segset_dir)

expected = '/content/data/FGADR-Seg-set_Release/Seg-set'
if fgadr_segset_dir != expected:
    os.makedirs(os.path.dirname(expected), exist_ok=True)
    if not os.path.exists(expected):
        os.symlink(fgadr_segset_dir, expected)
    print(f'Normalized path: {expected} -> {fgadr_segset_dir}')

shutil.copy(
    '/content/M2-DRProgression/module1/data/DR_Seg_Grading_Label_Filtered.csv',
    os.path.join(expected, 'DR_Seg_Grading_Label_Filtered.csv')
)
print('done')


In [ ]:
%cd /content/dr-joint-learning/dr_segmentation
!pip install -q segmentation-models-pytorch


In [ ]:
# Swap in our reduced-epoch, locally-pathed config
!cp /content/M2-DRProgression/module1/configs/config_fgadr_seg_poc.py config_fgadr.py
!cat config_fgadr.py


In [ ]:
# EX = hard exudates.
# Let this run to the end (all 25 epochs) -- do NOT press stop. No output between the
# "Starting epoch N/25" lines is normal at 1280px (a few min/epoch on a T4). To check it's
# alive without interrupting: open a scratch cell and run  !nvidia-smi  -- you want a python
# process listed with >0% GPU-Util. Checkpoints now land every 5 epochs, so a lost session
# after epoch >=5 still leaves a usable model in results/models_FGADR_NO_TATL_ex/.
!python train_fgadr.py --seed 765 --preprocess 2 --lesion EX


In [ ]:
# MA = microaneurysms.
# Same rules as cell 7: run to completion, don't interrupt. Run cell 9 BEFORE starting this
# one so the finished EX checkpoint is already banked to Drive if this session dies mid-MA.
!python train_fgadr.py --seed 765 --preprocess 2 --lesion MA


In [ ]:
# Persist checkpoints to Drive (train_fgadr.py saves under dr_segmentation/results/ on the
# Colab VM disk, which is wiped when the runtime recycles). Absolute path -- cell 10 %cd's
# away, so a relative 'results/' would resolve wrong if cells are re-run out of order.
import shutil, glob, os

SEG_TRAIN_DIR = '/content/dr-joint-learning/dr_segmentation'
src_dirs = sorted(glob.glob(os.path.join(SEG_TRAIN_DIR, 'results', 'models_FGADR_NO_TATL_*')))
assert src_dirs, (
    f"No {SEG_TRAIN_DIR}/results/models_FGADR_NO_TATL_* -- cells 7 and 8 (train_fgadr.py) "
    "did not finish. Scroll up and read their output for the real error. Then check:\n"
    f"    !ls -R {SEG_TRAIN_DIR}/results/"
)

dest = '/content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation'
os.makedirs(dest, exist_ok=True)
for d in src_dirs:
    ckpts = glob.glob(os.path.join(d, '*.pth.tar'))
    assert ckpts, f"{d} has no *.pth.tar -- training for that lesion crashed before saving."
    shutil.copytree(d, os.path.join(dest, os.path.basename(d)), dirs_exist_ok=True)
    print('Copied', os.path.basename(d), '->', sorted(os.path.basename(c) for c in ckpts))
print('Drive now holds:', os.listdir(dest))


In [ ]:
# Dice/IoU on top of the same predictions (manuscript's own committed metric, in addition
# to train_fgadr.py's own AP/ROC-AUC printed during training -- report both, see module1/README.md)
import glob, os

REPO         = '/content/M2-DRProgression/module1'
SEG_DIR      = '/content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation'
FGADR_ROOT   = '/content/data/FGADR-Seg-set_Release/Seg-set'
FILTERED_CSV = os.path.join(REPO, 'data', 'DR_Seg_Grading_Label_Filtered.csv')  # abs path -- script default is repo-root-relative
IMAGE_SIZE   = 1280  # match the resolution train_fgadr.py trains at (--preprocess 2 FGADR path), not the script's 512 default

def find_ckpt(lesion):
    have = os.listdir(SEG_DIR) if os.path.isdir(SEG_DIR) else []
    # train_fgadr.py lowercases the lesion for the dir name; match case-insensitively anyway
    dirs = [d for d in glob.glob(os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_*'))
            if d.lower().endswith('_' + lesion.lower())]
    assert dirs, f"No models_FGADR_NO_TATL_{lesion} in {SEG_DIR} -- run cells 7-9 first. Have: {have}"
    ckpts = glob.glob(os.path.join(dirs[0], '*.pth.tar'))
    assert ckpts, f"{dirs[0]} has no *.pth.tar"
    best = [c for c in ckpts if 'best' in os.path.basename(c).lower()]
    if best:
        return best[0]
    def epoch_num(p):
        s = ''.join(ch for ch in os.path.basename(p) if ch.isdigit())
        return int(s) if s else -1
    return sorted(ckpts, key=lambda c: (epoch_num(c), os.path.getmtime(c)))[-1]

for lesion in ('EX', 'MA'):
    ckpt = find_ckpt(lesion)
    print(f'\n=== {lesion}: {ckpt} ===')
    CURVE_OUT = f'/content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation/roc_curve_{lesion}.json'
    !python {REPO}/evaluate_segmentation_dice_iou.py --fgadr-root {FGADR_ROOT} --filtered-csv {FILTERED_CSV} --checkpoint "{ckpt}" --lesion {lesion} --image-size {IMAGE_SIZE} --curve-out "{CURVE_OUT}"


In [ ]:
# Task K/O figures: per-lesion ROC curve overlay + a qualitative segmentation grid
# (image | ground truth | predicted mask) for each trained lesion.
import os
os.makedirs('/content/drive/MyDrive/Thesis_Datasets/module1_runs/figures', exist_ok=True)
FIG_DIR = '/content/drive/MyDrive/Thesis_Datasets/module1_runs/figures'

curve_args = ' '.join(
    f'--curve {lesion}=/content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation/roc_curve_{lesion}.json'
    for lesion in ('EX', 'MA')
    if os.path.isfile(f'/content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation/roc_curve_{lesion}.json')
)
!python {REPO}/plot_figures.py roc {curve_args} --out {FIG_DIR}/module1_roc_curves.png

for lesion in ('EX', 'MA'):
    ckpt = find_ckpt(lesion)
    !python {REPO}/plot_figures.py seg-grid --fgadr-root {FGADR_ROOT} --filtered-csv {FILTERED_CSV} \
        --checkpoint "{ckpt}" --lesion {lesion} --n-examples 3 --image-size {IMAGE_SIZE} \
        --out {FIG_DIR}/module1_seg_grid_{lesion}.png
